# grad-tracking-global-toggle — worked example 3: set_grad_enabled(mode) — value-taking context manager

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `grad-tracking-global-toggle`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

Unlike `NoGrad` which always disables tracking, `set_grad_enabled(mode)` accepts an explicit boolean, allowing code to re-enable tracking inside an outer `NoGrad` block. This is useful for training loops that run a validation step mid-epoch: the outer context disables grad, but a specific inner block can re-enable it. The snapshot-and-restore pattern remains the same — save the previous value, set the new one, restore on exit.

## Worked solution

**Step 1 — store mode and snapshot on enter.** `__init__` saves the requested mode. `__enter__` reads the current global, stores it in `self.prev`, then sets the global to `self.mode`.

**Step 2 — restore on exit.** `__exit__` writes back `self.prev`. Returns `False` to propagate exceptions.

**Step 3 — demonstrate re-enable inside NoGrad.** With the outer flag set to `False` via `set_grad_enabled(False)`, an inner `set_grad_enabled(True)` temporarily re-enables tracking. After the inner block exits, tracking is disabled again (restored to `False`).

**Step 4 — verify nesting math.** Start True -> False -> True -> False -> True. Each transition is a snapshot/restore, not a counter — so the restore is always exactly the state that was active when `__enter__` ran.

In [ ]:
import torch as t

grad_tracking_enabled = True

def get_flag():
    return globals()['grad_tracking_enabled']

def set_flag(v):
    globals()['grad_tracking_enabled'] = v

class SetGradEnabled:
    def __init__(self, mode: bool):
        self.mode = bool(mode)
        self.prev = None
    def __enter__(self):
        self.prev = get_flag()
        set_flag(self.mode)
        return self
    def __exit__(self, exc_type, exc_val, tb):
        set_flag(self.prev)
        return False

# Exercise
print(f"Start: {get_flag()}")                    # True
with SetGradEnabled(False):
    print(f"  Outer disabled: {get_flag()}")     # False
    with SetGradEnabled(True):
        print(f"    Re-enabled: {get_flag()}")   # True
    print(f"  After inner: {get_flag()}")        # False (restored outer)
print(f"End: {get_flag()}")                      # True

assert get_flag() == True
print("Nesting check passed.")